# 005-village-crossdate — best and worst cases

# Imports

In [ ]:
import json
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import pandas as pd

from uavloc.data.drone_frames import DroneFrame, load_frames
from uavloc.data.satellite import GeoImage
from uavloc.features.factory import build_features, correlator_config
from uavloc.geo import LatLon
from uavloc.matching.correlator import PositionCorrelator
from uavloc.matching.replay import load_records, replay
from uavloc.plotting.correlation import plot_review
from uavloc.track import REPO_ROOT, experiment_dir, load_runs

# Params

In [ ]:
EXP = "005-village-crossdate"
RUN = "005.2"
VGG_RUN = "005.1"
NOISE_M = 15.0
N_CASES = 5
DEVICE = "mps"
OVERVIEW_MARGIN_M = 50.0

# Data

In [ ]:
config = next(r for r in load_runs(EXP) if r["run"] == RUN)["config"]
frames = {f.name: f for f in load_frames(config["frames_dataset"])}

In [ ]:
def extreme_cases(worst: bool) -> pd.DataFrame:
    """The N_CASES frames with the lowest (or highest) error at NOISE_M, one draw per frame."""
    records = load_records(EXP, RUN)
    vgg = load_records(EXP, VGG_RUN)[["frame", "noise_m", "bearing_deg", "error_m"]]
    draws = records.merge(vgg, on=["frame", "noise_m", "bearing_deg"], suffixes=("", "_vgg11"))
    draws = draws[draws["noise_m"] == NOISE_M].sort_values("error_m", ascending=not worst)
    return draws.drop_duplicates("frame").head(N_CASES).reset_index(drop=True)

In [ ]:
cases = pd.concat([extreme_cases(False).assign(kind="best"), extreme_cases(True).assign(kind="worst")], ignore_index=True)
cases[["kind", "place", "frame", "bearing_deg", "prior_error_m", "error_m", "error_m_vgg11", "peak_z", "reachable"]].round(2)

# Model

In [ ]:
def place_map(frame: DroneFrame) -> GeoImage:
    meta = json.loads((frame.image_path.parent / "meta.json").read_text())
    return GeoImage.load(REPO_ROOT / meta["map_image"], LatLon(*meta["map_top_left"]), LatLon(*meta["map_bottom_right"]))

In [ ]:
maps, correlators = {}, {}
for place, group in cases.groupby("place"):
    maps[place] = place_map(frames[group["frame"].iloc[0]])
    correlators[place] = PositionCorrelator(build_features(config, maps[place], DEVICE), correlator_config(config))

# Run

In [ ]:
replays = [replay(correlators[c.place], frames[c.frame], LatLon(c.prior_lat, c.prior_lon)) for c in cases.itertuples()]

# Plots

In [ ]:
for case, r in zip(cases.itertuples(), replays):
    out = experiment_dir(EXP) / "figures" / f"{case.kind}-cases"
    out.mkdir(parents=True, exist_ok=True)
    rank = case.Index % N_CASES + 1
    print(f"{case.kind.title()}: {case.frame}, Distance: {case.error_m:.2f} m | peak z: {case.peak_z:.2f}")
    map_date = json.loads((r.frame.image_path.parent / "meta.json").read_text())["map_date"]
    fig = plot_review(r, maps[case.place], f"{case.kind.upper()}: {case.frame} | estimate {case.prior_error_m:.1f} m off -> "
                      f"corrected {case.error_m:.2f} m (VGG11 {case.error_m_vgg11:.1f} m)", map_date, OVERVIEW_MARGIN_M)
    fig.savefig(out / f"{case.kind}-{rank}-{case.frame}.png", dpi=80)
    plt.show()